In [2]:
import random 
import pickle

import numpy as np 
import pandas as pd 

from nltk.tokenize import RegexpTokenizer

import torch.nn as nn


In [6]:
text_df=pd.read_csv("fake_or_real_news.csv")



In [8]:
text=list(text_df['text'].values)
joined_text=" ".join(text)

In [21]:
partial_text=joined_text[:100000]

In [22]:
tokenizer=RegexpTokenizer(r"\w+")#this means that we are only taking words for the tokenization process
tokens=tokenizer.tokenize(partial_text.lower())

In [23]:
tokens

['daniel',
 'greenfield',
 'a',
 'shillman',
 'journalism',
 'fellow',
 'at',
 'the',
 'freedom',
 'center',
 'is',
 'a',
 'new',
 'york',
 'writer',
 'focusing',
 'on',
 'radical',
 'islam',
 'in',
 'the',
 'final',
 'stretch',
 'of',
 'the',
 'election',
 'hillary',
 'rodham',
 'clinton',
 'has',
 'gone',
 'to',
 'war',
 'with',
 'the',
 'fbi',
 'the',
 'word',
 'unprecedented',
 'has',
 'been',
 'thrown',
 'around',
 'so',
 'often',
 'this',
 'election',
 'that',
 'it',
 'ought',
 'to',
 'be',
 'retired',
 'but',
 'it',
 's',
 'still',
 'unprecedented',
 'for',
 'the',
 'nominee',
 'of',
 'a',
 'major',
 'political',
 'party',
 'to',
 'go',
 'war',
 'with',
 'the',
 'fbi',
 'but',
 'that',
 's',
 'exactly',
 'what',
 'hillary',
 'and',
 'her',
 'people',
 'have',
 'done',
 'coma',
 'patients',
 'just',
 'waking',
 'up',
 'now',
 'and',
 'watching',
 'an',
 'hour',
 'of',
 'cnn',
 'from',
 'their',
 'hospital',
 'beds',
 'would',
 'assume',
 'that',
 'fbi',
 'director',
 'james',
 'c

In [24]:
unique_tokens=np.unique(tokens)
unique_token_index={token: idx for idx,token in enumerate(unique_tokens)}

In [25]:
unique_tokens

array(['0', '000', '1', ..., 'zarif', 'zero', 'zhou'],
      shape=(3707,), dtype='<U18')

In [26]:
n_words=10
input_words=[]
next_words=[]
for i in range(len(tokens)-n_words):
    input_words.append(tokens[i:i+n_words])
    next_words.append(tokens[i+n_words])

In [27]:
X=np.zeros((len(input_words),n_words,len(unique_tokens)),dtype=bool)
y=np.zeros((len(next_words),len(unique_tokens)),dtype=bool)

In [30]:
for i, words in enumerate(input_words):
    for j, word in enumerate(words):
        X[i,j,unique_token_index[word]]=1
        y[i,unique_token_index[next_words[i]]]=1

In [37]:
import torch
import torch.nn as nn

class TextGenerationModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128):
        super(TextGenerationModel, self).__init__()
        
        # PyTorch can stack LSTMs automatically using num_layers
        # batch_first=True makes input shape (batch, seq, feature)
        self.lstm = nn.LSTM(
            input_size=vocab_size, 
            hidden_size=hidden_dim, 
            num_layers=2,           # Replaces needing two separate LSTM layers
            batch_first=True
        )
        
        # 'Dense' in PyTorch is called 'Linear'
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        # Softmax layer (though often omitted if using CrossEntropyLoss)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Pass through LSTM
        out, (hidden, cell) = self.lstm(x)
        
        # Take the output of the last time step for the prediction
        out = out[:, -1, :] 
        
        # Pass through fully connected layer and softmax
        out = self.fc(out)
        # out = self.softmax(out)
        return out

# Initialize the model
model = TextGenerationModel(vocab_size=len(unique_tokens))

In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# 1. Convert your data (X, y) into PyTorch Tensors
# Assuming X and y are currently NumPy arrays
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32) # Assuming y is one-hot encoded

# 2. Replicate batch_size=128 and shuffle=True using DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# 3. Replicate .compile() (Loss function and Optimizer)
# CrossEntropyLoss combines categorical_crossentropy and softmax
criterion = nn.CrossEntropyLoss() 
optimizer = optim.RMSprop(model.parameters(), lr=0.001)

# Optional: Move to GPU if available for faster training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 4. Replicate .fit(epochs=10) via a manual training loop
epochs = 10

for epoch in range(epochs):
    model.train() # Set model to training mode
    
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for batch_X, batch_y in dataloader:
        # Move batch data to the same device as the model
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Step A: Clear old gradients
        optimizer.zero_grad()
        
        # Step B: Forward pass (make predictions)
        predictions = model(batch_X)
        
        # Step C: Calculate the loss
        loss = criterion(predictions, batch_y)
        
        # Step D: Backward pass (calculate gradients)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        # Step E: Update model weights
        optimizer.step()
        
        # --- Calculate Metrics (Accuracy) ---
        running_loss += loss.item()
        
        # Get the predicted class index (highest probability)
        _, predicted_classes = torch.max(predictions, dim=1)
        
        # If y is one-hot encoded, get the target class index
        if batch_y.dim() > 1:
            _, true_classes = torch.max(batch_y, dim=1)
        else:
            true_classes = batch_y
            
        total_samples += true_classes.size(0)
        correct_predictions += (predicted_classes == true_classes).sum().item()
        
    # Calculate epoch averages
    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = (correct_predictions / total_samples) * 100
    
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_accuracy:.2f}%")

Epoch [1/10] - Loss: 6.4627 - Accuracy: 6.40%
Epoch [2/10] - Loss: 6.2559 - Accuracy: 6.82%
Epoch [3/10] - Loss: 6.0994 - Accuracy: 7.25%
Epoch [4/10] - Loss: 5.9336 - Accuracy: 7.64%
Epoch [5/10] - Loss: 5.7580 - Accuracy: 8.30%
Epoch [6/10] - Loss: 5.5867 - Accuracy: 9.06%
Epoch [7/10] - Loss: 5.4210 - Accuracy: 9.92%
Epoch [8/10] - Loss: 5.2544 - Accuracy: 10.78%
Epoch [9/10] - Loss: 5.0859 - Accuracy: 11.80%
Epoch [10/10] - Loss: 4.9296 - Accuracy: 12.59%


In [42]:
torch.save(model.state_dict(),"mymodel.pth")

In [44]:
model=TextGenerationModel(vocab_size=len(unique_tokens))
model.load_state_dict(torch.load("mymodel.pth"))
model.eval()

TextGenerationModel(
  (lstm): LSTM(3707, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=3707, bias=True)
  (softmax): Softmax(dim=1)
)

In [47]:
import torch
import numpy as np

def predict_next_word(input_text, n_best):
    input_text = input_text.lower()
    
    # 1. Create the empty NumPy array
    X = np.zeros((1, n_words, len(unique_tokens)))
    
    # Optional but recommended: Only grab the last 'n_words' 
    # in case the user types a sentence longer than the model expects
    words = input_text.split()[-n_words:]
    
    # 2. Fill the array
    for i, word in enumerate(words):
        if word in unique_token_index:  # Prevents KeyError on unseen words
            X[0, i, unique_token_index[word]] = 1
            
    # 3. Convert NumPy array to PyTorch Tensor
    X_tensor = torch.tensor(X, dtype=torch.float32)
    
    # Optional: If you used a GPU for training, move the tensor to the GPU
    # X_tensor = X_tensor.to(device)
    
    # 4. Make the prediction
    model.eval() # Ensure dropout/training layers are locked
    with torch.no_grad():
        # Pass data directly into the model (replaces model.predict)
        predictions = model(X_tensor) 
        
    # 5. Get the indices of the highest scoring predictions
    # predictions is shape (1, vocab_size), so we look at [0]
    top_scores, top_indices = torch.topk(predictions[0], n_best)
    
    # Return as a standard Python list
    return top_indices.tolist()

In [49]:
# 1. First, make the prediction and assign it to the variable 'possible'
possible = predict_next_word("when we meet again we will have", 10)

# 2. Create a reverse dictionary so we can look up the words
index_to_word = {index: word for word, index in unique_token_index.items()}

# 3. Loop through the variable 'possible' and get the matching words
predicted_words = [index_to_word[idx] for idx in possible]

# 4. View the results
print("Input text: 'when we meet again we will have'")
print("Top 10 predictions:", predicted_words)

Input text: 'when we meet again we will have'
Top 10 predictions: [np.str_('than'), np.str_('i'), np.str_('to'), np.str_('a'), np.str_('so'), np.str_('as'), np.str_('the'), np.str_('on'), np.str_('about'), np.str_('women')]
